<a href="https://colab.research.google.com/github/RIYORIALDY/Optimization-from-Scratch/blob/main/ML_Optimization_from_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Implementasi Algoritma Optimasi dari Awal (From Scratch)

Notebook ini mendemonstrasikan implementasi algoritma optimasi yang umum digunakan dalam machine learning tanpa menggunakan library tingkat tinggi seperti `scikit-learn`. Fokus utamanya adalah memahami cara kerja algoritma tersebut dari dasar.

**Kasus yang dibahas:**
- **Regresi Logistik**: Untuk masalah klasifikasi biner (dua kelas).
- **Metode Optimasi**: Batch Gradient Descent (GD) dan Stochastic Gradient Descent (SGD).

**Tujuan:**
1.  Mengimplementasikan Regresi Logistik dari awal hanya dengan `NumPy`.
2.  Menerapkan dan membandingkan dua metode optimasi: Batch GD dan SGD.
3.  Mengevaluasi kinerja model menggunakan *loss function* (Binary Cross-Entropy).
4.  Memvisualisasikan hasil training, termasuk plot kurva loss dan confusion matrix, untuk membandingkan efisiensi kedua algoritma optimasi.

## 1. Import Library dan Pemuatan Data

Langkah pertama adalah mengimpor library yang diperlukan (`NumPy`, `Pandas`, `Matplotlib`, `Seaborn`) dan mengunduh dataset. Dataset yang digunakan adalah **Telco Customer Churn**, yang berisi informasi mengenai pelanggan perusahaan telekomunikasi dan status churn mereka (apakah mereka berhenti berlangganan atau tidak).

In [ ]:
# 1. IMPORT LIBRARY DAN DOWNLOAD DATA
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Download dataset
!wget -q https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv

df = pd.read_csv('Telco-Customer-Churn.csv')
print("Dataset shape:", df.shape)
print(df.head())

## 2. Pra-pemrosesan Data (Preprocessing)

Sebelum model dilatih, data perlu dipersiapkan melalui beberapa tahapan:

1.  **Seleksi Fitur**: Untuk menyederhanakan, kita hanya akan menggunakan tiga fitur numerik: `tenure`, `MonthlyCharges`, dan `TotalCharges`.
2.  **Penanganan Nilai Hilang**: Kolom `TotalCharges` memiliki beberapa nilai kosong yang diisi dengan nilai median dari kolom tersebut.
3.  **Konversi Target**: Variabel target `Churn` yang berisi 'Yes'/'No' diubah menjadi format biner (1/0).
4.  **Pembagian Data**: Dataset dibagi menjadi data latih (train) dan data uji (test) secara manual dengan perbandingan 80:20. Pembagian ini dilakukan secara *stratified* untuk memastikan proporsi kelas target (churn vs non-churn) tetap sama di kedua set data.
5.  **Standardisasi Fitur**: Fitur-fitur numerik distandardisasi (menggunakan Z-score scaling) agar memiliki rata-rata 0 dan standar deviasi 1. Ini penting agar algoritma Gradient Descent dapat berjalan lebih stabil dan efisien.

In [ ]:
# 2. PREPROCESSING DATA
numerical_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

X = df[numerical_features].values
y = (df['Churn'] == 'Yes').astype(int).values  # Convert Yes/No to 1/0

# Manual stratified train_test_split
def my_train_test_split(X, y, test_size=0.2, random_state=42):
    np.random.seed(random_state)
    idx0 = np.where(y == 0)[0]
    idx1 = np.where(y == 1)[0]
    np.random.shuffle(idx0)
    np.random.shuffle(idx1)
    n0_test = int(len(idx0) * test_size)
    n1_test = int(len(idx1) * test_size)
    test_idx = np.concatenate([idx0[:n0_test], idx1[:n1_test]])
    train_idx = np.concatenate([idx0[n0_test:], idx1[n1_test:]])
    np.random.shuffle(train_idx)
    np.random.shuffle(test_idx)
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

X_train, X_test, y_train, y_test = my_train_test_split(X, y, test_size=0.2, random_state=42)

# Manual standardisasi
X_mean = X_train.mean(axis=0)
X_std = X_train.std(axis=0)
X_train_scaled = (X_train - X_mean) / X_std
X_test_scaled = (X_test - X_mean) / X_std

## 3. Implementasi Model dari Awal

Di bagian ini, kita akan membuat tiga kelas model dari dasar:
1.  **Linear Regression (Batch GD)**: Sebagai pembanding sederhana.
2.  **Logistic Regression (Batch GD)**: Implementasi Regresi Logistik dengan optimasi Batch Gradient Descent.
3.  **Logistic Regression (SGD)**: Implementasi Regresi Logistik dengan optimasi Stochastic Gradient Descent.

### 3.1. Regresi Linear (Batch Gradient Descent)

Kelas ini mengimplementasikan regresi linear sederhana untuk memprediksi nilai kontinu. Meskipun tidak cocok untuk masalah klasifikasi ini, kelas ini berguna sebagai dasar untuk memahami cara kerja Gradient Descent. *Loss function* yang digunakan adalah **Mean Squared Error (MSE)**.

In [ ]:
# 3. LINEAR REGRESSION (Batch GD)
class LinearRegressionGD:
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.lr = learning_rate
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = None
        self.losses = []

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0
        for i in range(self.n_iterations):
            y_pred = np.dot(X, self.weights) + self.bias
            loss = np.mean((y_pred - y) ** 2)
            self.losses.append(loss)
            dw = (2/n_samples) * np.dot(X.T, (y_pred - y))
            db = (2/n_samples) * np.sum(y_pred - y)
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
            if (i+1) % 100 == 0:
                print(f"Linear GD iter {i+1}: loss={loss:.4f}")

    def predict(self, X):
        return np.dot(X, self.weights) + self.bias

    def score(self, X, y):
        y_pred = self.predict(X)
        ss_total = np.sum((y - np.mean(y)) ** 2)
        ss_residual = np.sum((y - y_pred) ** 2)
        return 1 - (ss_residual / ss_total)

### 3.2. Regresi Logistik (Batch Gradient Descent)

**Batch Gradient Descent (GD)** adalah metode optimasi yang menghitung gradien dari *loss function* terhadap seluruh data training pada setiap iterasi. Akibatnya, pembaruan parameter (bobot dan bias) dilakukan sekali per epoch.

- **Kelebihan**: Konvergensi lebih stabil dan mengarah langsung ke titik minimum.
- **Kekurangan**: Membutuhkan banyak memori dan komputasi yang mahal jika dataset sangat besar, karena seluruh data harus diproses sekaligus.

Fungsi *loss* yang digunakan adalah **Binary Cross-Entropy (BCE)**, yang cocok untuk masalah klasifikasi biner.

In [ ]:
# 4. LOGISTIC REGRESSION (Batch GD)
class LogisticRegressionGD:
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.lr = learning_rate
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = None
        self.losses = []

    def sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0
        for i in range(self.n_iterations):
            linear_pred = np.dot(X, self.weights) + self.bias
            y_pred = self.sigmoid(linear_pred)
            epsilon = 1e-15
            y_pred = np.clip(y_pred, epsilon, 1-epsilon)
            loss = -np.mean(y * np.log(y_pred) + (1-y) * np.log(1-y_pred))
            self.losses.append(loss)
            dw = (1/n_samples) * np.dot(X.T, (y_pred - y))
            db = (1/n_samples) * np.sum(y_pred - y)
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
            if (i+1) % 100 == 0:
                print(f"Logistic GD iter {i+1}: loss={loss:.4f}")

    def predict_proba(self, X):
        linear_pred = np.dot(X, self.weights) + self.bias
        return self.sigmoid(linear_pred)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

    def accuracy(self, X, y):
        y_pred = self.predict(X)
        return np.mean(y_pred == y)

### 3.3. Regresi Logistik (Stochastic Gradient Descent)

**Stochastic Gradient Descent (SGD)** adalah varian dari Gradient Descent di mana gradien dan pembaruan parameter dilakukan untuk **setiap satu data training** secara acak pada setiap epoch.

- **Kelebihan**: Komputasi jauh lebih cepat dan efisien untuk dataset besar. Fluktuasi yang terjadi selama training dapat membantu algoritma keluar dari minimum lokal.
- **Kekurangan**: Konvergensi tidak stabil (berfluktuasi) dan tidak pernah benar-benar mencapai titik minimum global, melainkan hanya mendekatinya. Ini terlihat dari kurva loss yang lebih "berisik".

In [ ]:
# 5. LOGISTIC REGRESSION SGD
class LogisticRegressionSGD:
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.lr = learning_rate
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = None
        self.losses = []

    def sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0
        for i in range(self.n_iterations):
            indices = np.random.permutation(n_samples)
            epoch_loss = 0
            for idx in indices:
                xi = X[idx]
                yi = y[idx]
                linear_pred = np.dot(xi, self.weights) + self.bias
                y_pred = self.sigmoid(linear_pred)
                epsilon = 1e-15
                y_pred = np.clip(y_pred, epsilon, 1-epsilon)
                sample_loss = -(yi*np.log(y_pred) + (1-yi)*np.log(1-y_pred))
                epoch_loss += sample_loss
                dw = xi * (y_pred - yi)
                db = y_pred - yi
                self.weights -= self.lr * dw
                self.bias -= self.lr * db
            avg_loss = epoch_loss / n_samples
            self.losses.append(avg_loss)
            if (i+1) % 100 == 0:
                print(f"Logistic SGD iter {i+1}: loss={avg_loss:.4f}")

    def predict_proba(self, X):
        linear_pred = np.dot(X, self.weights) + self.bias
        return self.sigmoid(linear_pred)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

    def accuracy(self, X, y):
        y_pred = self.predict(X)
        return np.mean(y_pred == y)

## 4. Pelatihan Model

Sekarang, kita akan melatih ketiga model yang telah dibuat pada data training yang sudah distandardisasi. Hyperparameter seperti `learning_rate` dan `n_iterations` diatur untuk masing-masing model. Setelah pelatihan, kinerja setiap model dievaluasi pada data training dan data testing menggunakan metrik yang sesuai (R-squared untuk Regresi Linear dan akurasi untuk Regresi Logistik).

In [ ]:
# 6. TRAINING MODELS
print("\nTraining Linear Regression:")
lr_model = LinearRegressionGD(learning_rate=0.01, n_iterations=500)
lr_model.fit(X_train_scaled, y_train)
print(f"Linear R2 (Train): {lr_model.score(X_train_scaled, y_train):.4f}")
print(f"Linear R2 (Test): {lr_model.score(X_test_scaled, y_test):.4f}")

print("\nTraining Logistic Regression (Batch GD):")
log_model_batch = LogisticRegressionGD(learning_rate=0.1, n_iterations=500)
log_model_batch.fit(X_train_scaled, y_train)
print(f"Logistic BatchGD Acc (Train): {log_model_batch.accuracy(X_train_scaled, y_train):.4f}")
print(f"Logistic BatchGD Acc (Test): {log_model_batch.accuracy(X_test_scaled, y_test):.4f}")

print("\nTraining Logistic Regression (SGD):")
log_model_sgd = LogisticRegressionSGD(learning_rate=0.01, n_iterations=500)
log_model_sgd.fit(X_train_scaled, y_train)
print(f"Logistic SGD Acc (Train): {log_model_sgd.accuracy(X_train_scaled, y_train):.4f}")
print(f"Logistic SGD Acc (Test): {log_model_sgd.accuracy(X_test_scaled, y_test):.4f}")

## 5. Visualisasi dan Analisis Hasil

Untuk membandingkan kinerja algoritma, kita akan memvisualisasikan:

1.  **Kurva Loss**: Plot ini menunjukkan bagaimana nilai *loss function* menurun seiring berjalannya iterasi. Ini membantu kita melihat kecepatan dan stabilitas konvergensi.
    - **Batch GD**: Kurva loss terlihat mulus dan stabil, menurun secara konsisten.
    - **SGD**: Kurva loss terlihat lebih berfluktuasi atau "berisik" karena pembaruan parameter dilakukan pada setiap sampel.
2.  **Confusion Matrix**: Matriks ini menunjukkan performa klasifikasi model pada data uji, membandingkan prediksi dengan nilai sebenarnya (True Positive, True Negative, False Positive, False Negative).

In [ ]:
# 7. VISUALISASI HASIL
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(lr_model.losses, color='blue', linewidth=2)
axes[0].set_title('Linear Regression - MSE Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Iteration'); axes[0].set_ylabel('Mean Squared Error')
axes[1].plot(log_model_batch.losses, color='green', linewidth=2)
axes[1].set_title('Logistic Regression (Batch GD) - BCE Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Iteration'); axes[1].set_ylabel('Binary Cross Entropy')
axes[2].plot(log_model_sgd.losses, color='red', linewidth=2)
axes[2].set_title('Logistic Regression (SGD) - BCE Loss', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Iteration'); axes[2].set_ylabel('Binary Cross Entropy')
plt.tight_layout()
plt.show()

# Confusion Matrix Plot
def plot_confusion_matrix(y_true, y_pred, title):
    cm = np.zeros((2,2), dtype=int)
    for yt, yp in zip(y_true, y_pred):
        cm[yt, yp] += 1
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No Churn', 'Churn'], yticklabels=['No Churn', 'Churn'])
    plt.title(title, fontsize=14, fontweight='bold')
    plt.ylabel('Actual'); plt.xlabel('Predicted')
    plt.show()

y_pred_batch = log_model_batch.predict(X_test_scaled)
plot_confusion_matrix(y_test, y_pred_batch, 'Conf. Matrix - Logistic Regression (Batch GD)')
y_pred_sgd = log_model_sgd.predict(X_test_scaled)
plot_confusion_matrix(y_test, y_pred_sgd, 'Conf. Matrix - Logistic Regression (SGD)')

## 6. Kesimpulan

Dari implementasi dan eksperimen di atas, dapat disimpulkan:

- **Implementasi dari Awal**: Berhasil mengimplementasikan Regresi Logistik dan dua varian Gradient Descent tanpa library ML tingkat tinggi.
- **Batch GD vs. SGD**:
  - **Kinerja**: Keduanya mencapai tingkat akurasi yang serupa pada data uji. Dengan hyperparameter yang tepat, SGD dapat seefektif Batch GD.
  - **Efisiensi**: SGD jauh lebih cepat dalam hal waktu komputasi per epoch, terutama pada dataset besar. Namun, konvergensinya lebih berisik, seperti yang ditunjukkan oleh plot loss.
  - **Stabilitas**: Batch GD menunjukkan konvergensi yang lebih mulus dan dapat diandalkan, meskipun lebih lambat.

Eksperimen ini memberikan pemahaman mendalam tentang bagaimana algoritma optimasi bekerja di balik layar, serta trade-off antara kecepatan, stabilitas, dan penggunaan memori.